# 05 · Model evaluation and comparison
**OMARI Seminar Series — From Pixels to Predictions: Developing AI Models for Medical Image Interpretation**

Load the saved test predictions of all three models and compare confusion matrices, per-class precision/recall/F1, calibration (with temperature fitted on validation), label efficiency, misclassifications, machine subgroups and paired statistical tests. Export machine-readable results for the website.

| | |
|---|---|
| **Runtime** | CPU is fine (minutes) |
| **Needs** | `runs/*/test_predictions.npz` from notebooks 02–04 |
| **Produces** | `website_export/{evaluation,training_curves,label_efficiency}.json` |
| **Modes** | `smoke` (synthetic data, random weights: checks the code only) · `demo` (real data subset, few epochs) · `full` (complete experiment) |

> Results from `smoke` mode are meaningless by construction and must never be exported to the website.
> Every artifact written by these notebooks records the script, checkpoint hash and parameters that produced it.

In [ ]:
#@title 0 · Setup: repository, packages, data and artifact folders { display-mode: "form" }
import os, sys, subprocess, json, random, time
MODE = "demo"  #@param ["smoke", "demo", "full"]
REPO_URL = "https://github.com/YOUR-ORG/omari-workshop.git"  #@param {type:"string"}
DATA_ROOT = "/content/data/FETAL_PLANES_ZENODO"  #@param {type:"string"}
ARTIFACTS = "/content/omari_artifacts"  #@param {type:"string"}
USE_DRIVE_FOR_ARTIFACTS = False  #@param {type:"boolean"}
MODE = os.environ.get("OMARI_MODE", MODE)            # lets CI / nbclient override
ARTIFACTS = os.environ.get("OMARI_ARTIFACTS", ARTIFACTS)
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB and USE_DRIVE_FOR_ARTIFACTS:
    from google.colab import drive
    drive.mount("/content/drive")
    ARTIFACTS = "/content/drive/MyDrive/omari_artifacts"

# 1) make `omari_us` importable: installed package > local checkout (../python) > clone REPO_URL
def _find_repo():
    for cand in [os.getcwd(), os.path.dirname(os.getcwd()), "/content/omari-workshop"]:
        if os.path.isdir(os.path.join(cand, "python", "omari_us")):
            return cand
    return None
REPO = _find_repo()
if REPO is None:
    if "YOUR-ORG" in REPO_URL:
        raise RuntimeError("Set REPO_URL to the repository that contains python/omari_us (or upload the repo to /content/omari-workshop).")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/omari-workshop"], check=True)
    REPO = "/content/omari-workshop"
sys.path.insert(0, os.path.join(REPO, "python"))

# 2) packages (Colab already has torch/torchvision; timm and friends are added)
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm>=0.9.12", "scikit-learn", "scipy", "pyyaml", "opencv-python-headless", "gdown"], check=True)

import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
import omari_us
from omari_us.utils import seed_everything, get_device, CLASSES, CLASS_SHORT
SEED = 42
seed_everything(SEED)
DEVICE = get_device()
os.makedirs(ARTIFACTS, exist_ok=True)
SPLITS_CSV = os.path.join(ARTIFACTS, "splits", "split_manifest.csv")
RUNS = os.path.join(ARTIFACTS, "runs")
print(f"omari_us {omari_us.__version__} | torch {torch.__version__} | device {DEVICE} | MODE={MODE} | artifacts → {ARTIFACTS}")
if DEVICE.type == "cpu" and MODE != "smoke":
    print("⚠ No GPU. In Colab: Runtime → Change runtime type → T4 GPU. CPU training of these models is impractically slow.")

In [ ]:
from omari_us.evaluation import metrics as M
import glob
RUN_FOR = {}
for mid in ["densenet161", "vit_b16", "usf_mae"]:
    cands = sorted(glob.glob(os.path.join(RUNS, f"{mid}_finetune_lf1_s0*")))
    if cands: RUN_FOR[mid] = cands[0]
print(RUN_FOR)
P = {m: dict(np.load(os.path.join(r, "test_predictions.npz"), allow_pickle=True)) for m, r in RUN_FOR.items()}
assert P, "No runs found — train with notebooks 02–04 first (same ARTIFACTS folder)."

## 1 · Headline metrics with patient-cluster bootstrap CIs
Resampling *patients* (not images) respects within-patient correlation.

In [ ]:
rows = []
for m, z in P.items():
    y, p = z["labels"].astype(int), z["probs"]
    r = M.classification_report(y, p.argmax(1)); ci = M.bootstrap_ci(y, p.argmax(1), groups=z["patients"], n_boot=1000 if MODE == "full" else 200)
    rel = M.reliability(p, y)
    rows.append({"model": m, "n": len(y), "accuracy": r["accuracy"], "acc 95% CI": np.round(ci["accuracy"], 3), "macro-F1": r["macro_f1"],
                 "F1 95% CI": np.round(ci["macro_f1"], 3), "balanced acc": r["balanced_accuracy"], "ECE": rel["ece"], "NLL": M.nll(p, y)})
pd.DataFrame(rows).set_index("model").round(3)

## 2 · Confusion matrices and per-class metrics

In [ ]:
fig, axes = plt.subplots(1, len(P), figsize=(5 * len(P), 4.4))
axes = np.atleast_1d(axes)
for ax, (m, z) in zip(axes, P.items()):
    cm = M.confusion(z["labels"].astype(int), z["probs"].argmax(1)); cmn = cm / cm.sum(1, keepdims=True).clip(1)
    ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    for i in range(6):
        for j in range(6): ax.text(j, i, f"{cmn[i,j]*100:.0f}", ha="center", va="center", color="white" if cmn[i, j] > .5 else "black", fontsize=8)
    ax.set_xticks(range(6), CLASS_SHORT, rotation=45); ax.set_yticks(range(6), CLASS_SHORT); ax.set_title(m); ax.set_xlabel("predicted"); ax.set_ylabel("true")
plt.tight_layout(); plt.show()
pc = {m: pd.DataFrame(M.classification_report(z["labels"].astype(int), z["probs"].argmax(1))["per_class"], index=CLASS_SHORT) for m, z in P.items()}
pd.concat(pc, axis=1).round(3)

## 3 · Calibration
Temperature is fitted on the **validation** predictions, then applied to test.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5)); ax.plot([0, 1], [0, 1], "k--", lw=1)
for m, z in P.items():
    y, p = z["labels"].astype(int), z["probs"]
    v = np.load(os.path.join(RUN_FOR[m], "val_predictions.npz"), allow_pickle=True)
    T = M.fit_temperature(v["probs"], v["labels"].astype(int))
    b = [r for r in M.reliability(p, y)["bins"] if r["count"]]
    ax.plot([r["conf"] for r in b], [r["acc"] for r in b], "o-", label=f"{m} (ECE {M.reliability(p, y)['ece']:.3f} → {M.reliability(M.apply_temperature(p, T), y)['ece']:.3f} at T={T:.2f})")
ax.set_xlabel("confidence"); ax.set_ylabel("accuracy"); ax.legend(fontsize=7); plt.show()

## 4 · Paired comparisons
Exact McNemar tests on the same test images, Holm-adjusted. Image-level pairing ignores patient clustering; treat p-values as approximate.

In [ ]:
ms = list(P); out = []
for i in range(len(ms)):
    for j in range(i + 1, len(ms)):
        a, b = P[ms[i]], P[ms[j]]; oa, ob = np.argsort(a["ids"]), np.argsort(b["ids"])
        assert (a["ids"][oa] == b["ids"][ob]).all()
        t = M.mcnemar(a["labels"][oa].astype(int), a["probs"][oa].argmax(1), b["probs"][ob].argmax(1)); out.append({"a": ms[i], "b": ms[j], **t})
if out:
    for o, adj in zip(out, M.holm([o["p_value"] for o in out])): o["p_holm"] = adj
pd.DataFrame(out)

## 5 · Domain shift by machine and the hardest errors

In [ ]:
for m, z in P.items():
    if "machines" in z and len(set(z["machines"])) > 1:
        print(m); display(pd.DataFrame(M.by_group(z["labels"].astype(int), z["probs"].argmax(1), z["machines"])).T.round(3))
m0 = list(P)[0]; z = P[m0]
conf = z["probs"].max(1); wrong = np.nonzero(z["probs"].argmax(1) != z["labels"])[0]
print(f"{m0}: {len(wrong)} errors; most confident errors:")
pd.DataFrame({"id": z["ids"][wrong], "true": [CLASS_SHORT[i] for i in z["labels"][wrong]], "pred": [CLASS_SHORT[i] for i in z["probs"][wrong].argmax(1)], "confidence": conf[wrong]}).sort_values("confidence", ascending=False).head(10)

## 6 · Label efficiency
Collected from `summary.json` files of the label-fraction sweeps (notebooks 02–04).

In [ ]:
S = [json.load(open(f)) for f in glob.glob(os.path.join(RUNS, "*", "summary.json"))]
le = pd.DataFrame([{"model": s["config"]["model"], "regime": s["config"]["regime"], "fraction": s["config"]["label_fraction"], "seed": s["config"]["seed"], "macro_f1": s["test"]["macro_f1"]} for s in S])
if le.fraction.nunique() > 1:
    g = le.groupby(["model", "regime", "fraction"]).macro_f1.agg(["mean", "min", "max"]).reset_index()
    for (m, rg), d in g.groupby(["model", "regime"]):
        plt.plot(d.fraction, d["mean"], "o-", label=f"{m} ({rg})"); plt.fill_between(d.fraction, d["min"], d["max"], alpha=.15)
    plt.xscale("log"); plt.xlabel("fraction of training labels (by patient)"); plt.ylabel("test macro-F1"); plt.legend(); plt.show()
else:
    print("Only one label fraction found; run the optional sweeps in notebooks 02–04.")

## 7 · Export for the website
Uses the same exporter as `scripts/evaluate.py`, so the website's numbers are exactly these.

In [ ]:
from omari_us.export import Exporter
from omari_us.datasets import load_split_manifest
cfg = {"output_dir": os.path.join(ARTIFACTS, "website_export"), "evaluation": {"runs": RUN_FOR, "bootstrap": 1000 if MODE == "full" else 200,
       "label_efficiency_glob": os.path.join(RUNS, "*", "summary.json"), "export_test_probabilities": True}}
ex = Exporter(cfg); ex.out.mkdir(parents=True, exist_ok=True)
splits_df = pd.read_csv(SPLITS_CSV)
print(ex.export_evaluation(splits_df), ex.export_curves(), ex.export_label_efficiency())
if MODE == "smoke": print("SMOKE MODE — do not copy these files to the website.")

## References
- Burgos-Artizzu X.P. et al. *Evaluation of deep convolutional neural networks for automatic classification of common maternal fetal ultrasound planes.* Sci. Rep. 10, 10200 (2020). Dataset: Zenodo 3904280, CC BY 4.0.
- Huang G. et al. *Densely connected convolutional networks.* CVPR 2017.
- Dosovitskiy A. et al. *An image is worth 16×16 words.* ICLR 2021; Steiner A. et al. *How to train your ViT?* TMLR 2022.
- He K. et al. *Masked autoencoders are scalable vision learners.* CVPR 2022.
- Megahed Y. et al. *USF-MAE: Ultrasound self-supervised foundation model with masked autoencoding.* Biomed. Signal Process. Control 122, 110313 (2026). https://github.com/Yusufii9/USF-MAE
- Zhou B. et al. CAM, CVPR 2016 · Selvaraju R.R. et al. Grad-CAM, ICCV 2017 · Chattopadhay A. et al. Grad-CAM++, WACV 2018 · Abnar & Zuidema, attention rollout, ACL 2020 · Chefer H. et al., CVPR 2021 · Adebayo J. et al., sanity checks, NeurIPS 2018 · Petsiuk V. et al., RISE / deletion–insertion, BMVC 2018 · Guo C. et al., calibration, ICML 2017.